# Module: SQLite3 Assignments
## Lesson: SQLite3

In [1]:
import sqlite3

### Assignment 1: Creating and Connecting to a Database

1. Write a Python function to create a new SQLite3 database named `test.db`.
2. Write a Python function to create a table named `employees` with columns `id` (integer), `name` (text), `age` (integer), and `department` (text) in the `test.db` database.

In [ ]:
conn = sqlite3.connect("database/test.db")

In [3]:
cursor = conn.cursor()

In [4]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS employees(
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        age INTEGER NOT NULL,
        department TEXT NOT NULL
    )
""")
conn.commit()

### Assignment 2: Inserting Data

1. Write a Python function to insert a new employee into the `employees` table.
2. Insert at least 5 different employees into the `employees` table.

In [8]:
employees = [
    ("Lin", 23, "engineering"),
    ("Ana", 31, "sales"),
    ("Raj", 28, "marketing"),
    ("Mei", 35, "engineering"),
    ("Tom", 26, "hr"),
]

cursor.executemany(
    """
        INSERT INTO employees(name, age, department)
        VALUES (?, ?, ?)
    """,
    employees,
)

conn.commit()

### Assignment 3: Querying Data

1. Write a Python function to fetch and display all records from the `employees` table.
2. Write a Python function to fetch and display all employees from a specific department.

In [9]:
cursor.execute("SELECT * FROM employees")
cursor.fetchall()

[(1, 'Lin', 23, 'engineering'),
 (2, 'Ana', 31, 'sales'),
 (3, 'Raj', 28, 'marketing'),
 (4, 'Mei', 35, 'engineering'),
 (5, 'Tom', 26, 'hr')]

In [10]:
cursor.execute("SELECT * FROM employees WHERE department = 'engineering'")
cursor.fetchall()

[(1, 'Lin', 23, 'engineering'), (4, 'Mei', 35, 'engineering')]

### Assignment 4: Updating Data

1. Write a Python function to update the department of an employee based on their `id`.
2. Update the department of at least 2 employees and display the updated records.

In [12]:
cursor.execute("UPDATE employees set department = 'hr' WHERE id = 4")
conn.commit()

cursor.execute("SELECT * FROM employees WHERE id = 4")
cursor.fetchall()

[(4, 'Mei', 35, 'hr')]

In [13]:
cursor.execute("UPDATE employees set department = 'engineering' WHERE name = 'Raj'")
cursor.execute("UPDATE employees set department = 'sales' WHERE id = 5")
conn.commit()

cursor.execute("SELECT * FROM employees WHERE id = 5 OR name = 'Raj'")
cursor.fetchall()

[(3, 'Raj', 28, 'engineering'), (5, 'Tom', 26, 'sales')]

### Assignment 5: Deleting Data

1. Write a Python function to delete an employee from the `employees` table based on their `id`.
2. Delete at least 1 employee and display the remaining records.

In [14]:
cursor.execute("DELETE FROM employees WHERE id = 5")
conn.commit()

cursor.execute("SELECT * FROM employees")
cursor.fetchall()

[(1, 'Lin', 23, 'engineering'),
 (2, 'Ana', 31, 'sales'),
 (3, 'Raj', 28, 'engineering'),
 (4, 'Mei', 35, 'hr')]

### Assignment 6: Advanced Queries

1. Write a Python function to fetch and display employees older than a certain age.
2. Write a Python function to fetch and display employees whose names start with a specific letter.

In [17]:
def older_than(age: int) -> list:
    cursor.execute("SELECT * FROM employees WHERE age > ?", (age,))
    return cursor.fetchall()

In [18]:
def name_starts_with(first_letter: str) -> list:
    cursor.execute("SELECT * FROM employees WHERE name LIKE ?", (first_letter + "%",))
    return cursor.fetchall()

### Assignment 7: Handling Transactions

1. Write a Python function to insert multiple employees into the `employees` table in a single transaction. Ensure that if any insertion fails, none of the insertions are committed.
2. Write a Python function to update the age of multiple employees in a single transaction. Ensure that if any update fails, none of the updates are committed.

In [ ]:
def add_employees(employees: list) -> None:
    try:
        cursor.executemany(
            """
                INSERT INTO employees(name, age, department)
                VALUES (?, ?, ?)
            """,
            employees,
        )
        conn.commit()
    except sqlite3.Error:
        conn.rollback()
        raise

In [ ]:
def update_age(details: list) -> None:
    try:
        for age, emp_id in details:
            cursor.execute("UPDATE employees SET age = ? WHERE id = ?", (age, emp_id))
            if cursor.rowcount == 0:
                raise sqlite3.Error(f"no employee with id {emp_id}")
        conn.commit()
    except sqlite3.Error:
        conn.rollback()
        raise

### Assignment 8: Creating Relationships

1. Create a new table named `departments` with columns `id` (integer) and `name` (text).
2. Modify the `employees` table to include a foreign key referencing the `id` column in the `departments` table.
3. Write a Python function to insert data into both the `departments` and `employees` tables, ensuring referential integrity.

In [ ]:
cursor.execute("PRAGMA foreign_keys = ON")
cursor.execute("CREATE TABLE IF NOT EXISTS departments(id INTEGER PRIMARY KEY, name TEXT)")
conn.commit()

In [ ]:
cursor.execute("ALTER TABLE employees ADD department_id INTEGER REFERENCES departments(id)")
conn.commit()

In [ ]:
def insert_into(dep_data: tuple, emp_data: list) -> None:
    """dep_data: (id, name), emp_data: [(name, age), ...]"""
    dep_id, dep_name = dep_data
    try:
        # department first, so the employees' department_id has a row to point at
        cursor.execute("INSERT INTO departments(id, name) VALUES (?, ?)", (dep_id, dep_name))
        cursor.executemany(
            "INSERT INTO employees(name, age, department, department_id) VALUES (?, ?, ?, ?)",
            [(name, age, dep_name, dep_id) for name, age in emp_data],
        )
        conn.commit()
    except sqlite3.Error:
        conn.rollback()
        raise

### Assignment 9: Indexing and Optimization

1. Create an index on the `name` column of the `employees` table.
2. Write a Python function to fetch and display all employees whose names start with a specific letter. Compare the performance with and without the index.

In [26]:
cursor.execute("CREATE INDEX name_index ON employees(name)")
conn.commit()

In [27]:
name_starts_with("R")

[(3, 'Raj', 28, 'engineering')]

### Assignment 10: Backing Up and Restoring Data

1. Write a Python function to back up the `test.db` database to a file named `backup.db`.
2. Write a Python function to restore the `test.db` database from the `backup.db` file.

In [ ]:
def backup_db(backup_path: str = "database/backup.db") -> None:
    backup_conn = sqlite3.connect(backup_path)
    conn.backup(backup_conn)
    backup_conn.close()

In [ ]:
def restore_db(backup_path: str = "database/backup.db") -> None:
    backup_conn = sqlite3.connect(backup_path)
    backup_conn.backup(conn)
    backup_conn.close()